In [ ]:
# Cell 1: Install required libraries

#!pip -q install SpeechRecognition ipywidgets


In [ ]:
# Cell 2: Import required libraries

import numpy as np
import scipy.signal as signal
import speech_recognition as sr
import ipywidgets as widgets

from IPython.display import Audio, display


In [ ]:
# Cell 3: Audio configuration

SAMPLE_RATE = 16000       # Samples per second
CHANNELS = 1              # Mono audio
RECORD_SECONDS = 5        # Duration for each test recording

print("Audio configuration:")
print("Sampling rate:", SAMPLE_RATE, "Hz")
print("Channels:", CHANNELS)
print("Test recording duration:", RECORD_SECONDS, "seconds")

In [ ]:
# Cell 4: Microphone input test

import base64
from google.colab import output

def record_microphone(duration=5):
    js_code = f"""
    async function recordAudio() {{
        const stream = await navigator.mediaDevices.getUserMedia({{audio: true}});

        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = event => {{
            if (event.data.size > 0) {{
                chunks.push(event.data);
            }}
        }};

        recorder.start();

        await new Promise(resolve => setTimeout(resolve, {duration * 1000}));

        recorder.stop();

        await new Promise(resolve => {{
            recorder.onstop = resolve;
        }});

        stream.getTracks().forEach(track => track.stop());

        const blob = new Blob(chunks, {{type: recorder.mimeType}});
        const buffer = await blob.arrayBuffer();

        let binary = '';
        const bytes = new Uint8Array(buffer);

        for (let i = 0; i < bytes.length; i++) {{
            binary += String.fromCharCode(bytes[i]);
        }}

        return btoa(binary);
    }}

    recordAudio();
    """

    audio_base64 = output.eval_js(js_code)

    audio_data = base64.b64decode(audio_base64)

    with open("/content/microphone_test.webm", "wb") as f:
        f.write(audio_data)

    return audio_data


print("Microphone test starting...")
print("Allow microphone access if your browser asks for permission.")
print("Speak normally for 5 seconds.")

audio_data = record_microphone(RECORD_SECONDS)

print("Microphone recording completed.")
print("Recorded data size:", len(audio_data), "bytes")

In [ ]:
# Cell 5: Audio playback test

print("Playing the recorded microphone audio...")

display(Audio("/content/microphone_test.webm"))

print("Playback control is shown above.")
print("Listen to the recording through your speaker/headphones.")

In [ ]:
# Cell 6: Voice command recognition
# Improved for "CSPML listen" and "CSPML stop"

import subprocess
import re
import difflib


def normalize_text(text):
    """Convert recognized speech into simple lowercase text."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def recognize_voice_command(duration=4):

    print("Listening...")
    print('Say: "CSPML listen" or "CSPML stop".')

    # Record microphone audio
    audio_data = record_microphone(duration)

    webm_file = "/content/voice_command.webm"
    wav_file = "/content/voice_command.wav"

    with open(webm_file, "wb") as f:
        f.write(audio_data)

    # Convert WebM to WAV
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i", webm_file,
            "-ar", str(SAMPLE_RATE),
            "-ac", "1",
            wav_file
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )

    recognizer = sr.Recognizer()

    try:
        with sr.AudioFile(wav_file) as source:
            audio = recognizer.record(source)

        # Use Indian English speech recognition
        results = recognizer.recognize_google(
            audio,
            language="en-IN",
            show_all=True
        )

        if not results or "alternative" not in results:
            print("No speech was recognized.")
            return ""

        alternatives = [
            normalize_text(item["transcript"])
            for item in results["alternative"]
        ]

        print("\nRecognized possibilities:")
        for text in alternatives:
            print("-", text)

        # Check all recognized possibilities
        for text in alternatives:

            words = text.split()

            # Look for the command word first.
            has_listen = any(
                difflib.SequenceMatcher(None, word, "listen").ratio() >= 0.70
                for word in words
            )

            has_stop = any(
                difflib.SequenceMatcher(None, word, "stop").ratio() >= 0.70
                for word in words
            )

            # Possible recognitions of "CSPML"
            cspml_variations = [
                "cspml",
                "csp ml",
                "c s p m l",
                "spml",
                "cspm",
                "sample"
            ]

            has_cspml = any(
                variation in text
                for variation in cspml_variations
            )

            # Detect LISTEN command
            if has_cspml and has_listen:
                print("\nCommand detected: CSPML LISTEN")
                return "cspml listen"

            # Detect STOP command
            if has_cspml and has_stop:
                print("\nCommand detected: CSPML STOP")
                return "cspml stop"

        print("\nCSPML command was not detected.")
        return ""

    except sr.UnknownValueError:
        print("Could not understand the speech.")
        return ""

    except sr.RequestError as error:
        print("Speech recognition service error:", error)
        return ""


command = recognize_voice_command()


In [ ]:
# Cell 7: Start/Stop state control

# Initial system state
system_state = "IDLE"

# Check the command recognized in Cell 6
if command == "cspml listen":
    system_state = "ACTIVE"

elif command == "cspml stop":
    system_state = "IDLE"

else:
    print("No valid CSPML command detected.")

print("System state:", system_state)

In [ ]:
# Cell 8: Noise estimation

import subprocess
from scipy.io import wavfile

print("Noise estimation recording starting...")
print("Stay SILENT for the first 1 second.")
print("Then speak normally for the remaining 4 seconds.")

# Record 5 seconds from the microphone
noise_recording = record_microphone(5)

webm_noise = "/content/noise_estimation.webm"
wav_noise = "/content/noise_estimation.wav"

# Save browser recording
with open(webm_noise, "wb") as f:
    f.write(noise_recording)

# Convert WebM to WAV
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", webm_noise,
        "-ar", str(SAMPLE_RATE),
        "-ac", "1",
        wav_noise
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=True
)

# Read WAV audio
sample_rate, audio_data = wavfile.read(wav_noise)

# Convert to floating-point values
audio_data = audio_data.astype(np.float32)

# Take the first 1 second as the noise sample
noise_samples = audio_data[:SAMPLE_RATE]

# Calculate FFT of the noise
noise_fft = np.fft.rfft(noise_samples)

# Estimate the average noise magnitude
noise_magnitude = np.abs(noise_fft)

print("\nNoise estimation completed.")
print("Sampling rate:", sample_rate, "Hz")
print("Noise samples used:", len(noise_samples))
print("Noise frequency bins:", len(noise_magnitude))

In [ ]:
# Cell 9: Noise cancellation

def noise_cancel(audio, noise_magnitude, reduction_factor=0.5):
    """
    Simple frequency-domain noise reduction.

    audio            : input audio signal x[n]
    noise_magnitude  : estimated noise magnitude from Cell 8
    reduction_factor : amount of noise reduction
    """

    # FFT of the input audio
    audio_fft = np.fft.rfft(audio)

    # Magnitude and phase of the input
    audio_magnitude = np.abs(audio_fft)
    audio_phase = np.angle(audio_fft)

    # Make the noise estimate the same size as the audio spectrum
    noise_estimate = np.interp(
        np.linspace(0, 1, len(audio_magnitude)),
        np.linspace(0, 1, len(noise_magnitude)),
        noise_magnitude
    )

    # Reduce frequency components related to the estimated noise
    cleaned_magnitude = np.maximum(
        audio_magnitude - reduction_factor * noise_estimate,
        0
    )

    # Reconstruct the complex spectrum
    cleaned_fft = cleaned_magnitude * np.exp(1j * audio_phase)

    # Convert back to the time domain
    cleaned_audio = np.fft.irfft(
        cleaned_fft,
        n=len(audio)
    )

    return cleaned_audio


# Use the recorded audio from Cell 8
input_audio = audio_data

# Apply noise cancellation
cleaned_audio = noise_cancel(
    input_audio,
    noise_magnitude
)

# Normalize to prevent excessive amplitude
max_value = np.max(np.abs(cleaned_audio))

if max_value > 0:
    cleaned_audio = cleaned_audio / max_value

print("Noise cancellation completed.")
print("Input samples:", len(input_audio))
print("Output samples:", len(cleaned_audio))

In [ ]:
# Cell 10: Test noise-cancelled audio

# Convert cleaned audio to 16-bit PCM
cleaned_audio_int16 = np.int16(
    np.clip(cleaned_audio, -1, 1) * 32767
)

# Save the cleaned audio as WAV
from scipy.io.wavfile import write

cleaned_wav_file = "/content/cleaned_audio.wav"

write(
    cleaned_wav_file,
    SAMPLE_RATE,
    cleaned_audio_int16
)

print("Playing noise-cancelled audio...")

display(Audio(cleaned_wav_file))

print("Noise-cancelled audio player is ready.")

In [ ]:
# Noise Cancellation Comparison Test

print("======================================")
print(" NOISE CANCELLATION COMPARISON TEST")
print("======================================")

# --------------------------------------------------
# Record one audio sample
# --------------------------------------------------

print("\nRecording for 7 seconds...")
print("Speak normally and include some background noise if possible.")

webm_data = record_microphone(duration=7)

# --------------------------------------------------
# Save browser recording
# --------------------------------------------------

webm_file = "/content/noise_test_original.webm"
original_wav = "/content/noise_test_original.wav"

with open(webm_file, "wb") as f:
    f.write(webm_data)

# --------------------------------------------------
# Convert WebM to WAV
# --------------------------------------------------

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", webm_file,
        "-ar", str(SAMPLE_RATE),
        "-ac", "1",
        original_wav
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=True
)

# --------------------------------------------------
# Read original audio
# --------------------------------------------------

sample_rate, original_audio = wavfile.read(
    original_wav
)

original_audio = original_audio.astype(np.float32)

# --------------------------------------------------
# Create noise-cancelled version
# --------------------------------------------------

noise_cancelled_audio = noise_cancel(
    original_audio,
    noise_magnitude
)

# --------------------------------------------------
# Normalize noise-cancelled audio
# --------------------------------------------------

max_value = np.max(
    np.abs(noise_cancelled_audio)
)

if max_value > 0:
    noise_cancelled_audio = (
        noise_cancelled_audio / max_value
    )

# --------------------------------------------------
# Convert both to 16-bit
# --------------------------------------------------

original_int16 = np.int16(
    np.clip(
        original_audio /
        max(np.max(np.abs(original_audio)), 1),
        -1,
        1
    ) * 32767
)

noise_cancelled_int16 = np.int16(
    np.clip(
        noise_cancelled_audio,
        -1,
        1
    ) * 32767
)

# --------------------------------------------------
# Save both versions
# --------------------------------------------------

noise_cancelled_wav = (
    "/content/noise_test_cancelled.wav"
)

wavfile.write(
    original_wav,
    SAMPLE_RATE,
    original_int16
)

wavfile.write(
    noise_cancelled_wav,
    SAMPLE_RATE,
    noise_cancelled_int16
)

# --------------------------------------------------
# PLAY ORIGINAL
# --------------------------------------------------

print("\n==============================")
print("1. ORIGINAL AUDIO")
print("==============================")

display(
    Audio(
        original_wav,
        autoplay=False
    )
)

# --------------------------------------------------
# PLAY NOISE-CANCELLED
# --------------------------------------------------

print("\n==============================")
print("2. NOISE-CANCELLED AUDIO")
print("==============================")

display(
    Audio(
        noise_cancelled_wav,
        autoplay=False
    )
)

print("\nComparison complete.")

In [ ]:
# Cell 11: Echo impulse response

# Echo parameters
echo_delay = 0.3       # Echo delay in seconds
echo_gain = 0.5        # Echo volume

# Convert delay from seconds to samples
delay_samples = int(echo_delay * SAMPLE_RATE)

# Create the impulse response
echo_ir = np.zeros(delay_samples + 1)

# Original impulse
echo_ir[0] = 1.0

# Delayed impulse
echo_ir[delay_samples] = echo_gain

print("Echo impulse response created.")
print("Echo delay:", echo_delay, "seconds")
print("Echo gain:", echo_gain)
print("Impulse response length:", len(echo_ir), "samples")

In [ ]:
# Cell 12: Echo convolution

# x[n] = input audio
x = cleaned_audio

# h[n] = echo impulse response
h = echo_ir

# Convolution: y[n] = x[n] * h[n]
echo_audio = signal.fftconvolve(x, h, mode="full")

# Normalize the output
max_value = np.max(np.abs(echo_audio))

if max_value > 0:
    echo_audio = echo_audio / max_value

# Convert to 16-bit audio
echo_audio_int16 = np.int16(
    np.clip(echo_audio, -1, 1) * 32767
)

# Save the echo output
echo_wav_file = "/content/echo_audio.wav"

write(
    echo_wav_file,
    SAMPLE_RATE,
    echo_audio_int16
)

print("Echo convolution completed.")
print("Input samples:", len(x))
print("Impulse response samples:", len(h))
print("Output samples:", len(echo_audio))

print("\nPlaying echo audio...")
display(Audio(echo_wav_file))

In [ ]:
# Cell 13: Drum beat impulse response

# Duration of the drum pattern
drum_duration = 1.0       # seconds

# Time between drum impulses
drum_interval = 0.25      # seconds

# Convert time values to samples
drum_length = int(drum_duration * SAMPLE_RATE)
drum_interval_samples = int(drum_interval * SAMPLE_RATE)

# Create the impulse response
drum_ir = np.zeros(drum_length)

# Create short impulses at regular intervals
drum_positions = range(
    0,
    drum_length,
    drum_interval_samples
)

for position in drum_positions:
    drum_ir[position] = 1.0

# Make later impulses slightly quieter
if len(drum_ir) > drum_interval_samples:
    drum_ir[drum_interval_samples] = 0.7

if len(drum_ir) > 2 * drum_interval_samples:
    drum_ir[2 * drum_interval_samples] = 0.5

if len(drum_ir) > 3 * drum_interval_samples:
    drum_ir[3 * drum_interval_samples] = 0.7

print("Drum beat impulse response created.")
print("Duration:", drum_duration, "seconds")
print("Beat interval:", drum_interval, "seconds")
print("Impulse response length:", len(drum_ir), "samples")

In [ ]:
# Cell 14: Drum beat convolution

# x[n] = cleaned input audio
x = cleaned_audio

# h[n] = drum beat impulse response
h = drum_ir

# Convolution
drum_audio = signal.fftconvolve(x, h, mode="full")

# Normalize the output
max_value = np.max(np.abs(drum_audio))

if max_value > 0:
    drum_audio = drum_audio / max_value

# Convert to 16-bit audio
drum_audio_int16 = np.int16(
    np.clip(drum_audio, -1, 1) * 32767
)

# Save the drum-effect output
drum_wav_file = "/content/drum_audio.wav"

write(
    drum_wav_file,
    SAMPLE_RATE,
    drum_audio_int16
)

print("Drum beat convolution completed.")
print("Input samples:", len(x))
print("Impulse response samples:", len(h))
print("Output samples:", len(drum_audio))

print("\nPlaying drum beat effect...")
display(Audio(drum_wav_file))

In [ ]:
# Cell 15: Effect selection GUI

# Create the effect selection dropdown
effect_selector = widgets.Dropdown(
    options=[
        "No Effect",
        "Echo",
        "Drum Beat"
    ],
    value="No Effect",
    description="Effect:"
)

# Display the GUI
display(effect_selector)

print("Select an audio effect from the dropdown.")

In [ ]:
import numpy as np

def noise_cancel(audio, noise_magnitude):
    """
    Simple noise reduction.

    Keeps the original voice signal and only suppresses
    very low-level signals below the estimated noise level.
    """

    audio = audio.astype(np.float32)

    # Estimate noise threshold
    threshold = noise_magnitude * 0.5

    # Keep the signal above the threshold
    output = np.where(
        np.abs(audio) > threshold,
        audio,
        audio * 0.1
    )

    return output.astype(np.float32)

print("New noise cancellation function loaded.")

In [ ]:
"""# Cell 16: Final system integration

import subprocess
from scipy.io import wavfile


# --------------------------------------------------
# Record one audio chunk
# --------------------------------------------------

def record_audio_chunk(duration=7):

    audio_data = record_microphone(duration)

    webm_file = "/content/current_audio.webm"
    wav_file = "/content/current_audio.wav"

    with open(webm_file, "wb") as f:
        f.write(audio_data)

    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i", webm_file,
            "-ar", str(SAMPLE_RATE),
            "-ac", "1",
            wav_file
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )

    sample_rate, recorded_audio = wavfile.read(wav_file)

    return recorded_audio.astype(np.float32)


# --------------------------------------------------
# Safer noise cancellation
# --------------------------------------------------

def noise_cancel(audio, noise_magnitude):

    audio = audio.astype(np.float32)

    # Convert noise magnitude array into one noise-level value
    noise_level = float(np.mean(np.abs(noise_magnitude)))

    # Use a small fraction of the estimated noise level
    threshold = noise_level * 0.05

    # Suppress only very low-level signals
    processed_audio = np.where(
        np.abs(audio) > threshold,
        audio,
        audio * 0.1
    )

    return processed_audio.astype(np.float32)

# --------------------------------------------------
# Process audio
# --------------------------------------------------

def process_audio_chunk(audio_data, selected_effect):

    audio = audio_data.astype(np.float32)

    max_value = np.max(np.abs(audio))

    if max_value > 0:
        audio = audio / max_value

    # Noise cancellation
    processed_audio = noise_cancel(
        audio,
        noise_magnitude
    )

    # Apply selected impulse-response effect
    if selected_effect == "Echo":

        processed_audio = signal.fftconvolve(
            processed_audio,
            echo_ir,
            mode="full"
        )

    elif selected_effect == "Drum Beat":

        processed_audio = signal.fftconvolve(
            processed_audio,
            drum_ir,
            mode="full"
        )

    # Normalize output
    max_value = np.max(np.abs(processed_audio))

    if max_value > 0:
        processed_audio = processed_audio / max_value

    return processed_audio


# --------------------------------------------------
# FINAL SYSTEM
# --------------------------------------------------

print("======================================")
print(" CSPML VOICE-ACTIVATED AUDIO SYSTEM")
print("======================================")

system_state = "IDLE"

print("\nSystem state:", system_state)
print('Say "CSPML listen" to start.')

# Voice command
start_command = recognize_voice_command(duration=4)

# Remove extra spaces and make lowercase
start_command = start_command.strip().lower()

if start_command == "cspml listen":

    system_state = "ACTIVE"

    print("\nCSPML listen detected.")
    print("System state:", system_state)

    # Read selected GUI effect
    selected_effect = effect_selector.value

    print("Selected effect:", selected_effect)
    print("\nAudio processing started.")
    print("Each audio chunk will be 7 seconds.")

    # Record one 7-second chunk
    print("\nRecording microphone audio...")
    current_audio = record_audio_chunk(duration=7)

    print("Recording completed.")

    # Process audio
    processed_audio = process_audio_chunk(
        current_audio,
        selected_effect
    )

    # Convert to 16-bit audio
    processed_audio_int16 = np.int16(
        np.clip(processed_audio, -1, 1) * 32767
    )

    # Save processed output
    output_file = "/content/final_processed_audio.wav"

    wavfile.write(
        output_file,
        SAMPLE_RATE,
        processed_audio_int16
    )

    # Play processed audio
    print("Playing processed audio...")
    display(Audio(output_file, autoplay=True))

    # Return to IDLE
    system_state = "IDLE"

    print("\nAudio processing chunk completed.")
    print("System state:", system_state)

else:

    print("\nCSPML listen was not detected.")
    print("System remains in IDLE state.")"""

In [ ]:
# Cell 16: Audio processing functions
# This cell only defines functions.
# It does not record, play, or print any audio.

# --------------------------------------------------
# Noise cancellation
# --------------------------------------------------

def noise_cancel(audio, noise_magnitude):

    audio = audio.astype(np.float32)

    # Convert the estimated noise spectrum
    # into one noise-level value
    noise_level = float(
        np.mean(np.abs(noise_magnitude))
    )

    # Small threshold for noise reduction
    threshold = noise_level * 0.05

    # Reduce very low-level components
    processed_audio = np.where(
        np.abs(audio) > threshold,
        audio,
        audio * 0.1
    )

    return processed_audio.astype(np.float32)


# --------------------------------------------------
# Audio processing
# --------------------------------------------------

def process_audio_chunk(audio_data, selected_effect):

    # Convert input audio to floating point
    audio = audio_data.astype(np.float32)

    # Normalize input
    max_value = np.max(np.abs(audio))

    if max_value > 0:
        audio = audio / max_value

    # -------------------------------
    # Noise cancellation
    # -------------------------------

    processed_audio = noise_cancel(
        audio,
        noise_magnitude
    )

    # -------------------------------
    # Selected impulse-response effect
    # -------------------------------

    if selected_effect == "Echo":

        processed_audio = signal.fftconvolve(
            processed_audio,
            echo_ir,
            mode="full"
        )

    elif selected_effect == "Drum Beat":

        processed_audio = signal.fftconvolve(
            processed_audio,
            drum_ir,
            mode="full"
        )

    # If "No Effect" is selected,
    # the noise-cancelled audio remains unchanged.

    # -------------------------------
    # Normalize output
    # -------------------------------

    max_value = np.max(
        np.abs(processed_audio)
    )

    if max_value > 0:
        processed_audio = (
            processed_audio / max_value
        )

    return processed_audio

In [ ]:
# Cell 17: Final start/stop system test
# CSPML stop is detected DURING the 7-second recording

import base64
import subprocess
from scipy.io import wavfile
from google.colab import output
from IPython.display import Audio, display, Javascript
import numpy as np


# --------------------------------------------------
# Record audio + listen for STOP simultaneously
# --------------------------------------------------

def record_audio_with_stop_detection(duration=7):

    js_code = r"""
    async function recordWithStopDetection(duration) {

        const stream = await navigator.mediaDevices.getUserMedia({
            audio: true
        });

        const recorder = new MediaRecorder(stream);
        const audioChunks = [];

        let stopDetected = false;
        let recordingFinished = false;

        recorder.ondataavailable = function(event) {
            if (event.data && event.data.size > 0) {
                audioChunks.push(event.data);
            }
        };

        const SpeechRecognition =
            window.SpeechRecognition ||
            window.webkitSpeechRecognition;

        let recognition = null;

        if (SpeechRecognition) {

            recognition = new SpeechRecognition();

            recognition.continuous = true;
            recognition.interimResults = true;
            recognition.lang = "en-US";

            recognition.onresult = function(event) {

                let text = "";

                for (
                    let i = event.resultIndex;
                    i < event.results.length;
                    i++
                ) {
                    text += event.results[i][0].transcript + " ";
                }

                text = text.toLowerCase().trim();

                console.log("Heard:", text);

                if (
                    text.includes("cspml stop") ||
                    text.includes("cspmlstop") ||
                    text.includes("csp ml stop")
                ) {

                    stopDetected = true;

                    if (recorder.state !== "inactive") {
                        recorder.stop();
                    }
                }
            };

            recognition.onerror = function(event) {
                console.log(
                    "Speech recognition error:",
                    event.error
                );
            };
        }

        // ------------------------------------------
        // Start recording
        // ------------------------------------------

        recorder.start(250);

        if (recognition) {
            try {
                recognition.start();
            } catch(e) {}
        }

        // ------------------------------------------
        // Wait for 7 seconds OR STOP command
        // ------------------------------------------

        await new Promise(resolve => {

            const timer = setTimeout(() => {

                if (recorder.state !== "inactive") {
                    recorder.stop();
                }

            }, duration * 1000);

            recorder.onstop = function() {

                clearTimeout(timer);

                recordingFinished = true;

                try {
                    if (recognition) {
                        recognition.stop();
                    }
                } catch(e) {}

                // Give MediaRecorder time to finish
                // its final dataavailable event.
                setTimeout(resolve, 200);
            };
        });

        // ------------------------------------------
        // Stop microphone
        // ------------------------------------------

        stream.getTracks().forEach(
            track => track.stop()
        );

        // ------------------------------------------
        // Create WebM blob
        // ------------------------------------------

        const blob = new Blob(
            audioChunks,
            {type: "audio/webm"}
        );

        console.log(
            "Recorded chunks:",
            audioChunks.length
        );

        console.log(
            "Recorded blob size:",
            blob.size
        );

        // ------------------------------------------
        // Convert blob to Base64
        // ------------------------------------------

        const buffer = await blob.arrayBuffer();

        const bytes = new Uint8Array(buffer);

        let binary = "";

        const chunkSize = 0x8000;

        for (
            let i = 0;
            i < bytes.length;
            i += chunkSize
        ) {

            binary += String.fromCharCode.apply(
                null,
                bytes.subarray(
                    i,
                    Math.min(i + chunkSize, bytes.length)
                )
            );
        }

        return {
            audio: btoa(binary),
            stopDetected: stopDetected,
            blobSize: blob.size
        };
    }

    recordWithStopDetection(7);
    """

    result = output.eval_js(js_code)

    audio_data = base64.b64decode(result["audio"])

    stop_detected = result["stopDetected"]

    blob_size = result["blobSize"]

    print("Browser recording size:", blob_size, "bytes")

    return audio_data, stop_detected


# --------------------------------------------------
# Convert WebM to WAV
# --------------------------------------------------

def convert_webm_to_audio(webm_data):

    webm_file = "/content/current_audio.webm"
    wav_file = "/content/current_audio.wav"

    # Save browser recording
    with open(webm_file, "wb") as f:
        f.write(webm_data)

    # Make sure recording is not empty
    if len(webm_data) == 0:
        raise ValueError(
            "Browser returned an empty audio recording."
        )

    print("Converting recorded audio to WAV...")

    # Convert WebM → WAV
    result = subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i", webm_file,
            "-ar", str(SAMPLE_RATE),
            "-ac", "1",
            wav_file
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    # Show ffmpeg error if conversion fails
    if result.returncode != 0:

        print("FFmpeg error:")
        print(
            result.stderr.decode(
                "utf-8",
                errors="ignore"
            )
        )

        raise RuntimeError(
            "FFmpeg could not convert the browser recording."
        )

    sample_rate, recorded_audio = wavfile.read(
        wav_file
    )

    return recorded_audio.astype(np.float32)


# --------------------------------------------------
# FINAL SYSTEM
# --------------------------------------------------

print("======================================")
print(" FINAL CSPML AUDIO SYSTEM TEST")
print("======================================")

system_state = "IDLE"

print("\nSystem state:", system_state)
print('Say "CSPML listen" to start.')


# --------------------------------------------------
# Wait for START command
# --------------------------------------------------

start_command = recognize_voice_command(
    duration=4
)

start_command = start_command.strip().lower()

if start_command == "cspml listen":

    system_state = "ACTIVE"

    print("\nCSPML listen detected.")
    print("System state:", system_state)

    selected_effect = effect_selector.value

    print("Selected effect:", selected_effect)

    # --------------------------------------------------
    # ACTIVE LOOP
    # --------------------------------------------------

    while system_state == "ACTIVE":

        print("\n--------------------------------------")
        print("Recording microphone audio...")
        print('Listening for "CSPML stop" simultaneously.')
        print("Recording duration: 7 seconds")
        print("--------------------------------------")

        # Record + listen for STOP at the same time
        webm_data, stop_detected = (
            record_audio_with_stop_detection(
                duration=7
            )
        )

        # ------------------------------------------
        # STOP detected
        # ------------------------------------------

        if stop_detected:

            system_state = "IDLE"

            print("\nCSPML stop detected.")
            print("Current audio chunk discarded.")
            print("Audio processing stopped.")
            print("System state:", system_state)

            break

        # ------------------------------------------
        # Convert recording to WAV
        # ------------------------------------------

        current_audio = convert_webm_to_audio(
            webm_data
        )

        print("Recording completed.")

        # ------------------------------------------
        # Process audio
        # ------------------------------------------

        processed_audio = process_audio_chunk(
            current_audio,
            selected_effect
        )

        # ------------------------------------------
        # Convert to 16-bit audio
        # ------------------------------------------

        processed_audio_int16 = np.int16(
            np.clip(
                processed_audio,
                -1,
                1
            ) * 32767
        )

        # ------------------------------------------
        # Save processed audio
        # ------------------------------------------

        output_file = (
            "/content/final_processed_audio.wav"
        )

        wavfile.write(
            output_file,
            SAMPLE_RATE,
            processed_audio_int16
        )

        # ------------------------------------------
        # Play processed audio
        # ------------------------------------------

        print("Playing processed audio...")

        display(
            Audio(
                output_file,
                autoplay=True
            )
        )

        print("\nSystem remains ACTIVE.")
        print(
            "The next 7-second recording "
            "will begin automatically."
        )

else:

    print("\nCSPML listen was not detected.")
    print("System remains IDLE.")